In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder,PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [21]:
data=pd.read_csv("loan.csv")
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333.0,1516.0,95.0,360.0,1.0,Urban,Y


In [22]:
data.duplicated().sum()

np.int64(0)

In [23]:
data.isna().sum()

Loan_ID               0
Gender                5
Married               0
Dependents            8
Education             4
Self_Employed        23
ApplicantIncome       4
CoapplicantIncome     5
LoanAmount            2
Loan_Amount_Term     13
Credit_History       30
Property_Area         1
Loan_Status           0
dtype: int64

In [24]:
data.dropna(axis=0,inplace=True)

In [25]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298 entries, 0 to 378
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            298 non-null    object 
 1   Gender             298 non-null    object 
 2   Married            298 non-null    object 
 3   Dependents         298 non-null    object 
 4   Education          298 non-null    object 
 5   Self_Employed      298 non-null    object 
 6   ApplicantIncome    298 non-null    float64
 7   CoapplicantIncome  298 non-null    float64
 8   LoanAmount         298 non-null    float64
 9   Loan_Amount_Term   298 non-null    float64
 10  Credit_History     298 non-null    float64
 11  Property_Area      298 non-null    object 
 12  Loan_Status        298 non-null    object 
dtypes: float64(5), object(8)
memory usage: 32.6+ KB


In [26]:
data.isna().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [27]:
# dropping columns which has no connection to outcome
data.drop(columns=['Loan_ID', 'Dependents', 'Married'], inplace=True)

In [28]:
X=data.drop("Loan_Status",axis=1)
y=data["Loan_Status"]

In [29]:
X

,Gender,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,Male,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural
1,Male,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban
2,Male,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban
3,Male,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban
4,Male,Not Graduate,No,2333.0,1516.0,95.0,360.0,1.0,Urban
...,...,...,...,...,...,...,...,...,...
373,Male,Graduate,No,3859.0,3300.0,142.0,180.0,1.0,Rural
374,Male,Not Graduate,No,3833.0,0.0,110.0,360.0,1.0,Rural
376,Male,Graduate,No,5703.0,0.0,128.0,360.0,1.0,Urban
377,Male,Graduate,No,3232.0,1950.0,108.0,360.0,1.0,Rural


In [30]:
y

0      N
1      Y
2      Y
3      Y
4      Y
      ..
373    Y
374    Y
376    Y
377    Y
378    Y
Name: Loan_Status, Length: 298, dtype: object

In [31]:
from sklearn.model_selection import train_test_split # x k andr multiple col thas why it s capital and need for randm is kiuk kahi dataset m shur s yes na bad m no ranome
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [32]:
from sklearn.linear_model import LogisticRegression

In [33]:
log_reg1=LogisticRegression()

In [34]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object))
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",LogisticRegression())
])

In [35]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
data.columns

Index(['Gender', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History',
       'Property_Area', 'Loan_Status'],
      dtype='object')

In [37]:
def predictor(Gender, Education, Self_Employed, ApplicantIncome, CoapplicantIncome,
              LoanAmount, Loan_Amount_Term, Credit_History, Property_Area):
    input_data = pd.DataFrame([{
        'Gender': Gender,
        'Education': Education,
        'Self_Employed': Self_Employed,
        'ApplicantIncome': ApplicantIncome,
        'CoapplicantIncome': CoapplicantIncome,
        'LoanAmount': LoanAmount,
        'Loan_Amount_Term': Loan_Amount_Term,
        'Credit_History': Credit_History,
        'Property_Area': Property_Area
    }])

    pred = pipeline.predict(input_data)[0]

    if pred == 'Y':
        print("loan mil gya bhai, party de")
    else:
        print("loan reject hogya, kal fir try karna")

In [38]:
predictor("Male", "Graduate", "No", 5000, 1500, 128, 360, 1.0, "Urban")

loan mil gya bhai, party de
